In [ ]:
#%% [markdown]
# # Q-Learning 逃離 21×11 迷宮（含 5 個寶藏 + Mask State + Early Stop）

#%%
import numpy as np
import random

# ─── 一、參數與地圖設定 ─────────────────────────────────────────
WIDTH, HEIGHT = 21, 11
NUM_POS = WIDTH * HEIGHT           # 231

# 題目給的寶藏座標
treasure_list = [
    (0, 6),   # index 6
    (3, 16),  # index 79
    (8, 2),   # index 170
    (10, 2),  # index 212
    (10, 17)  # index 227
]
treasures = set(treasure_list)

# 題目給的牆壁座標
walls_coords = [
    (0,4),(0,5),(0,7),(0,9),(1,1),(1,2),(1,4),(1,9),(1,10),(1,14),(1,18),
    (2,1),(2,3),(2,5),(2,7),(2,8),(2,9),(2,11),(2,13),(2,15),(2,16),(2,17),(2,19),
    (3,2),(3,8),(3,11),(3,17),(4,1),(4,4),(4,6),(4,10),(4,13),(4,16),(4,17),(4,18),(4,20),
    (5,4),(5,5),(5,6),(5,8),(5,9),(5,14),(5,15),(6,1),(6,2),(6,3),(6,6),(6,8),(6,10),(6,15),(6,16),(6,17),(6,19),
    (7,4),(7,6),(7,8),(7,10),(7,11),(7,17),(7,19),(8,1),(8,4),(8,8),(8,10),(8,13),(8,15),(8,18),(8,19),
    (9,1),(9,2),(9,4),(9,6),(9,7),(9,17),(10,1),(10,4),(10,16),(10,19)
]
walls = set(walls_coords)

START = (0,0)
GOAL  = (20,10)

# Q-learning 超參數
MAX_EPISODES = 3000
MAX_STEPS    = 1000

EPSILON   = 0.9
EPS_DECAY  = 0.999
EPS_MIN    = 0.01
ALPHA      = 0.5
GAMMA      = 0.995

# Early stop
patience = 500
no_improve = 0

# 動作空間
ACTIONS = ['up','down','left','right']
NUM_ACTIONS = len(ACTIONS)

# Mask 狀態：位置 231 × 2^5 = 32
# 寶箱拿取的狀態
NUM_MASK   = 1 << len(treasure_list)
NUM_STATES = NUM_POS * NUM_MASK

# 初始化 Q-table
Q = np.zeros((NUM_STATES, NUM_ACTIONS))


# 工具函式：座標 ↔ 索引
def to_index(pos):
    x,y = pos
    return y * WIDTH + x

def to_pos(idx):
    return (idx % WIDTH, idx // WIDTH)

# 編碼 state = pos_index * NUM_MASK + mask
def encode_state(pos, mask):
    return to_index(pos) * NUM_MASK + mask

# 計算嘗試移動到哪裡（邊界 clamp）
def compute_target(pos, action):
    x,y = pos
    if action == 'up':
        y = max(0, y-1)
    elif action == 'down':
        y = min(HEIGHT-1, y+1)
    elif action == 'left':
        x = max(0, x-1)
    elif action == 'right':
        x = min(WIDTH-1, x+1)
    return (x,y)

#%% [markdown]
# ## Q-Learning 訓練迴圈（包含撞牆偵測、Mask 更新、Early Stop）

#%%
best_path  = None
best_steps = MAX_STEPS + 1
best_score = -1
epsilon    = EPSILON

for episode in range(1, MAX_EPISODES+1):
    pos     = START
    mask    = 0
    state   = encode_state(pos, mask)
    path    = [state]
    steps   = 0
    score   = 0

    for step in range(1, MAX_STEPS+1):
        # ε-貪婪選動作
        if random.random() < epsilon:
            a = random.randrange(NUM_ACTIONS)
        else:
            a = np.argmax(Q[state])
        action = ACTIONS[a]

        # 1) 嘗試移動目標
        attempted = compute_target(pos, action)

        # 2) 撞牆/邊界判定
        if attempted in walls or attempted == pos:
            new_pos, new_mask = pos, mask
            r = -10
        else:
            new_pos, new_mask = attempted, mask

            # 3) 寶藏獎勵
            if new_pos in treasures:
                i = treasure_list.index(new_pos)
                if not (mask & (1<<i)):
                    new_mask |= (1<<i)
                    r = +10
                    score += 1
                else:
                    r = -1
            # 4) 終點獎勵（需收集全寶）
            elif new_pos == GOAL:
                if new_mask == (NUM_MASK-1):
                    r = +58
                else:
                    r = -50
            # 5) 一般步數懲罰
            else:
                r = -1

        # 6) 更新 state
        new_state = encode_state(new_pos, new_mask)

        # 7) Q-value 更新
        q_pred   = Q[state, a]
        if new_pos == GOAL and new_mask == (NUM_MASK-1):
            q_tgt = r
        else:
            q_tgt = r + GAMMA * np.max(Q[new_state])
        Q[state, a] += ALPHA * (q_tgt - q_pred)

        # 8) 更新 pos/mask/state & 路徑
        pos, mask, state = new_pos, new_mask, new_state
        path.append(state)
        steps += 1

        # 成功條件：全寶+終點
        if new_pos == GOAL and new_mask == (NUM_MASK-1):
            break

    # ε 衰減
    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)

    # 記錄最佳表現
    improved = False
    if pos == GOAL and mask == (NUM_MASK-1) and steps < best_steps:
        best_steps, best_score, best_path = steps, score, path.copy()
        improved = True

    # Early stop 檢查
    if improved:
        no_improve = 0
    else:
        no_improve += 1
    if epsilon <= EPS_MIN and no_improve >= patience:
        print(f"Early stop at episode {episode} (no improvement for {no_improve} eps).")
        break

    # 定期輸出
    if episode % 100 == 0:
        print(f'Ep{episode:4d} | ε={epsilon:.3f} | best_steps={best_steps}')

#%% [markdown]
# ## 輸出 & 儲存結果

#%%
if best_path is None:
    print("未找到收集 5 寶並到終點的路徑")
else:
    print("=== 最佳結果 ===")
    print(f"步數：{best_steps}，寶藏：{best_score}（應為 {len(treasure_list)}）")
    np.save('q_table.npy', Q)
    print("已儲存 Q-table → q_table.npy")

    # 文字化路徑顯示
    maze = [[' ']*WIDTH for _ in range(HEIGHT)]
    for x,y in walls_coords:  maze[y][x] = 'X'
    for x,y in treasure_list: maze[y][x] = 'O'
    sx,sy = START; gx,gy = GOAL
    maze[sy][sx] = 'S'; maze[gy][gx] = 'G'
    for s in best_path:
        pos_idx = s // NUM_MASK
        x,y = to_pos(pos_idx)
        if maze[y][x] == ' ':
            maze[y][x] = '.'
    print("\n路徑（. 為走過的路徑）：")
    for row in maze:
        print(''.join(row))

# EOF

Ep 100 | ε=0.814 | best_steps=1001
Ep 200 | ε=0.737 | best_steps=1001
Ep 300 | ε=0.667 | best_steps=1001
Ep 400 | ε=0.603 | best_steps=1001
Ep 500 | ε=0.546 | best_steps=1001
Ep 600 | ε=0.494 | best_steps=1001
Ep 700 | ε=0.447 | best_steps=1001
Ep 800 | ε=0.404 | best_steps=1001
Ep 900 | ε=0.366 | best_steps=1001
Ep1000 | ε=0.331 | best_steps=1001
Ep1100 | ε=0.299 | best_steps=1001
Ep1200 | ε=0.271 | best_steps=1001
Ep1300 | ε=0.245 | best_steps=1001
Ep1400 | ε=0.222 | best_steps=1001
Ep1500 | ε=0.201 | best_steps=1001
Ep1600 | ε=0.182 | best_steps=1001
Ep1700 | ε=0.164 | best_steps=1001
Ep1800 | ε=0.149 | best_steps=1001
Ep1900 | ε=0.134 | best_steps=1001
Ep2000 | ε=0.122 | best_steps=1001
Ep2100 | ε=0.110 | best_steps=1001
Ep2200 | ε=0.100 | best_steps=1001
Ep2300 | ε=0.090 | best_steps=1001
Ep2400 | ε=0.082 | best_steps=1001
Ep2500 | ε=0.074 | best_steps=1001
Ep2600 | ε=0.067 | best_steps=1001
Ep2700 | ε=0.060 | best_steps=1001
Ep2800 | ε=0.055 | best_steps=1001
Ep2900 | ε=0.049 | b